# 02 · Hide the answers (train/test split)

**Step T3 of the project plan.** To test a recommender fairly, we hide part of what each user really did and check whether it can guess it back,
like covering the answers of a quiz. This notebook builds **no recommender**: it prepares the exam that every recommender (T4, T6, T9) will take.

- **train** = what a recommender may look at.
- **test** = the hidden answers, used only to score a recommender.

Run notebook 01 first (it makes the data). Same seed, same split, every time.

## 0 · Settings and data

We load only the columns a recommender may see: the hidden ones (`interests`, `is_fake`, `topic`) are never read here.

In [1]:
import sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 0            # same seed, same split, every run
FRAC = 0.2          # hide 20% of each tested user's groups...
MIN_GROUPS = 3      # ...but only for users with at least 3 groups
TODAY, NEW_DAYS, SMALL = 365, 30, 10   # same meanings as in notebook 01

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'sim_v1'

# In Colab, the first time: upload sim_v1.zip (downloaded at the end of notebook 01)
if 'google.colab' in sys.modules and not (DATA / 'memberships.csv').exists():
    from google.colab import files
    uploaded = files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall(ROOT / 'data')

memberships = pd.read_csv(DATA / 'memberships.csv')
users = pd.read_csv(DATA / 'users.csv', usecols=['user_id', 'suspended'])                                  # never 'interests' or 'is_fake'
groups = pd.read_csv(DATA / 'groups.csv', usecols=['group_id', 'name', 'created_day', 'hidden_by_mods'])   # never 'topic'
posts = pd.read_csv(DATA / 'posts.csv')
likes = pd.read_csv(DATA / 'likes.csv')
print(f'{len(memberships)} memberships, {len(users)} users, {len(groups)} groups, {len(posts)} posts, {len(likes)} likes')

7959 memberships, 2000 users, 150 groups, 1080 posts, 7273 likes


## 1 · Who is tested, and what can be hidden

- **Tested users:** everyone with **3 or more groups**, except suspended accounts (they don't get suggestions). Users with 1 or 2 groups would have
  almost nothing left to learn from; newcomers get their own test later (experiment E5).
- **How many groups are hidden:** 20% of the user's groups, rounded, **at least 1**. That's 1 group for users with 3–7 groups, 2 for 8–12.
- **What can be hidden:** any membership except in groups hidden by moderators. A recommender must never suggest those (rule R8),
  so hiding them would test something no method is allowed to do.

The split is written as a function, so it can be reused later (see "What's next").

In [2]:
def split_memberships(m, frac=FRAC, min_groups=MIN_GROUPS, seed=SEED):
    # Hide part of each tested user's memberships. Returns (train, test): train stays visible, test is hidden.
    rng = np.random.default_rng(seed)
    n_groups = m.groupby('user_id').size()
    suspended = set(users.user_id[users.suspended == 1])
    off_limits = set(groups.group_id[groups.hidden_by_mods == 1])
    hideable = m[~m.group_id.isin(off_limits)]
    hidden_rows = []
    for uid, rows in hideable.groupby('user_id'):            # users in id order, so the result never changes
        n = int(n_groups[uid])
        if n < min_groups or uid in suspended:
            continue
        k = min(max(1, round(frac * n)), len(rows))           # 20%, rounded, at least 1
        hidden_rows += list(rng.choice(rows.index, size=k, replace=False))
    test = m.loc[sorted(hidden_rows)]
    train = m.drop(index=test.index)
    return train.reset_index(drop=True), test.reset_index(drop=True)

train, test = split_memberships(memberships)
print(f'{test.user_id.nunique()} users tested: {len(test)} memberships hidden, {len(train)} still visible')

1406 users tested: 1531 memberships hidden, 6428 still visible


## 2 · An example

One tested user with 5 groups. The recommender will only see the visible ones, and must guess the hidden one.

In [3]:
name = groups.set_index('group_id').name
uid = test.user_id[test.user_id.map(memberships.groupby('user_id').size()) == 5].iloc[0]
print(f'User {uid}')
print('  visible to the recommender:', ', '.join(name[train.group_id[train.user_id == uid]]))
print('  hidden (the answer)       :', ', '.join(name[test.group_id[test.user_id == uid]]))

User 1
  visible to the recommender: Finals countdown, Journaling streak, Korean conversation club, Midterms countdown
  hidden (the answer)       : Entrance exams accountability group


## 3 · Stop posts and likes from giving the answers away

Hiding a membership isn't enough. You can only post in a group you're in, so a post by a user in a hidden group would tell the recommender
they are a member. Likes on posts inside a group do the same (in the fake community, only members like group posts).
So we make **training copies of posts and likes without those rows**. Posts outside groups (`group_id = 0`) are kept.

In [4]:
hidden_pairs = set(zip(test.user_id, test.group_id))
leaky_post = np.array([(u, g) in hidden_pairs for u, g in zip(posts.user_id, posts.group_id)])
train_posts = posts[~leaky_post].copy()

post_group = posts.set_index('post_id').group_id
leaky_like = np.array([(u, g) in hidden_pairs for u, g in zip(likes.user_id, likes.post_id.map(post_group))])
train_likes = likes[~leaky_like & likes.post_id.isin(train_posts.post_id)].copy()
train_posts['likes'] = train_posts.post_id.map(train_likes.groupby('post_id').size()).fillna(0).astype(int)

print(f'posts: {leaky_post.sum()} removed, {len(train_posts)} kept | likes: {len(likes) - len(train_likes)} removed, {len(train_likes)} kept')

posts: 144 removed, 936 kept | likes: 2309 removed, 4964 kept


## 4 · Save the files

In [5]:
OUT = {'train_memberships.csv': train, 'test_memberships.csv': test,
       'train_posts.csv': train_posts, 'train_likes.csv': train_likes}
for fname, df in OUT.items():
    df.to_csv(DATA / fname, index=False)
    print(f'{fname:22} {len(df):6} rows')

train_memberships.csv    6428 rows
test_memberships.csv     1531 rows
train_posts.csv           936 rows
train_likes.csv          4964 rows


## 5 · Checks

T3 is done when there are two files and no membership is in both. We check that, and a few more things. A red error means something is wrong.

In [6]:
tr, te = pd.read_csv(DATA / 'train_memberships.csv'), pd.read_csv(DATA / 'test_memberships.csv')
tp, tl = pd.read_csv(DATA / 'train_posts.csv'), pd.read_csv(DATA / 'train_likes.csv')
pairs = lambda df: set(zip(df.user_id, df.group_id))

n_groups = memberships.groupby('user_id').size()
off_limits = set(groups.group_id[groups.hidden_by_mods == 1])
n_hideable = memberships[~memberships.group_id.isin(off_limits)].groupby('user_id').size()
suspended = users.set_index('user_id').suspended
eligible = {u for u in n_groups.index if n_groups[u] >= MIN_GROUPS and not suspended[u] and n_hideable.get(u, 0) > 0}
tested = te.user_id.unique()
expected = pd.Series({u: min(max(1, round(FRAC * int(n_groups[u]))), int(n_hideable[u])) for u in tested})
like_pairs = set(zip(tl.user_id, tl.post_id.map(tp.set_index('post_id').group_id)))
again_train, again_test = split_memberships(memberships)

checks = {
    'no membership is in both files': not (pairs(tr) & pairs(te)),
    'together they are exactly the original memberships':
        len(tr) + len(te) == len(memberships) and (pairs(tr) | pairs(te)) == pairs(memberships),
    'every user with 3+ groups (not suspended) is tested, and nobody else': set(tested) == eligible,
    'each tested user has 20% of their groups hidden (at least 1)': (te.groupby('user_id').size()[expected.index] == expected).all(),
    'every tested user still has at least 2 visible groups': tr.groupby('user_id').size()[tested].min() >= 2,
    'no group hidden by moderators is in the test': not te.group_id.isin(off_limits).any(),
    'training posts and likes give no hidden membership away': not (pairs(tp[tp.group_id > 0]) & pairs(te)) and not (like_pairs & pairs(te)),
    'like counts match the training likes': (tp.set_index('post_id').likes == tl.groupby('post_id').size().reindex(tp.post_id, fill_value=0)).all(),
    'same seed, same split': again_train.equals(train) and again_test.equals(test),
}
for check, ok in checks.items():
    print('✓' if ok else '✗', check)
assert all(checks.values()), 'Some checks failed (see ✗ above)'

✓ no membership is in both files
✓ together they are exactly the original memberships
✓ every user with 3+ groups (not suspended) is tested, and nobody else
✓ each tested user has 20% of their groups hidden (at least 1)
✓ every tested user still has at least 2 visible groups
✓ no group hidden by moderators is in the test
✓ training posts and likes give no hidden membership away
✓ like counts match the training likes
✓ same seed, same split


## 6 · What the exam looks like

In [7]:
size_train = tr.groupby('group_id').size().reindex(groups.group_id, fill_value=0)
created = groups.set_index('group_id').created_day
small_or_new = (size_train < SMALL) | (created > TODAY - NEW_DAYS)
top10 = set(size_train.nlargest(10).index)
per_user = te.groupby('user_id').size().value_counts()

summary = pd.Series({
    'tested users': f'{len(tested)} of {len(users)} ({len(tested) / len(users):.0%})',
    'hidden memberships (the answers)': len(te),
    'visible memberships (the training data)': len(tr),
    'tested users with 1 / 2 / 3 hidden groups': ' / '.join(str(per_user.get(k, 0)) for k in [1, 2, 3]),
    'hidden groups among the 10 biggest': f'{te.group_id.isin(top10).mean():.0%}',
    f'hidden groups that are small (< {SMALL} members) or new': f'{te.group_id.map(small_or_new).mean():.1%}',
})
print(summary.to_string())

tested users                                          1406 of 2000 (70%)
hidden memberships (the answers)                                    1531
visible memberships (the training data)                             6428
tested users with 1 / 2 / 3 hidden groups                 1281 / 125 / 0
hidden groups among the 10 biggest                                   33%
hidden groups that are small (< 10 members) or new                  2.7%


A ranking by size (T4) can only find hidden groups among the biggest ones. The small and new ones above are invisible to it: T5 will measure that.

## What's next

**The files a recommender may read**, from now on:

| A recommender **may** read | A recommender must **never** read |
|---|---|
| `train_memberships.csv`, `train_posts.csv`, `train_likes.csv` | `test_memberships.csv` (only to score) |
| `friendships.csv`, `blocks.csv`, `topics.csv`, `not_interested.csv` | `memberships.csv`, `posts.csv`, `likes.csv` (the full versions contain the answers) |
| `users.csv` and `groups.csv`, without the hidden columns | `interests`, `is_fake`, `topic` (the answer key) |

**T4 (the first recommender, popularity):** count group sizes from `train_memberships.csv`; rank groups that are open and not hidden by moderators
by size; for each tested user, take the 5 biggest they're not in; compare with `test_memberships.csv` (Precision@5, Recall@5, NDCG@5).

**Choosing settings** (for example the number of hidden tastes in T9) by looking at the test scores would be peeking. To tune, split the training
data again, `split_memberships(train, seed=1)`, tune on that, and keep the test file for the final scores only.

**Colab only:** the next cell downloads the four new files as a zip. Locally it does nothing.

In [8]:
if 'google.colab' in sys.modules:
    from google.colab import files
    with zipfile.ZipFile('sim_v1_split.zip', 'w') as z:
        for f in OUT:
            z.write(DATA / f, f'sim_v1/{f}')
    files.download('sim_v1_split.zip')
else:
    print('Running locally: the files are already in data/sim_v1')

Running locally: the files are already in data/sim_v1
